# 🎭 페르소나 파인튜닝 — Unsloth × Gemma (Colab 무료)

> 내 디지털 분신 팀(코라/핀/오피/리나/지오)을 LoRA로 학습시킨다.
> **이 노트북 하나로 5명 전부 학습 가능** — 맨 위 `AGENT` 변수만 바꾸면 됨.

**런타임 → 런타임 유형 변경 → T4 GPU** 먼저 선택!

단계: ①설치 → ②모델로드 → ③**학습 전 베이스라인** → ④데이터 → ⑤LoRA → ⑥학습 → ⑦**테스트(되는지 확인)** → ⑧저장

## 0. 설정 — 학습할 에이전트 선택

In [ ]:
# 학습할 에이전트 (코라부터 시작 권장)
AGENT = "cora"   # cora | finn | offie | rina | geo

# 강의 기준 모델(Gemma E2B). Unsloth 공식 노트북의 최신 모델명을 그대로 써도 됨.
MODEL_NAME = "unsloth/gemma-3n-E2B-it"
MAX_SEQ_LEN = 1024

# 데이터 파일명 (아래 셀에서 직접 업로드)
DATA_FILE = f"{AGENT}.jsonl"
print(f"▶ 학습 대상: {AGENT}  |  모델: {MODEL_NAME}")

## 1. Unsloth 설치

In [ ]:
%%capture
!pip install unsloth
# 최신판으로 갱신 (구버전 캐시 방지)
!pip install --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo

## 2. 모델 로드 (4bit 양자화)
`load_in_4bit=True` = 압축해서 무료 T4에서도 돌아가게. `full_finetuning=False` = LoRA(끝의 작은 행렬만 학습).

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,      # 양자화 ON
    full_finetuning = False,  # LoRA
)

## 3. 🔍 학습 전 베이스라인 — "아직 코라가 아님"을 확인
학습 전엔 페르소나가 없어야 정상. 학습 후 같은 질문을 다시 던져 **변화**를 비교한다.

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

def ask(model, question, max_new_tokens=128):
    FastModel.for_inference(model)
    messages = [{"role": "user", "content": question}]
    ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    )
    # ⚠️ 함정①: <bos> 중복 방지 — 학습 형식과 추론 형식을 맞춘다
    if ids[0, 0].item() == tokenizer.bos_token_id and ids[0, 1].item() == tokenizer.bos_token_id:
        ids = ids[:, 1:]
    ids = ids.to(model.device)
    out = model.generate(
        input_ids=ids, max_new_tokens=max_new_tokens,
        temperature=0.7, top_p=0.95, top_k=64,
    )
    text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    return text.strip()

print("[학습 전]", ask(model, "넌 누구야?"))

## 4. 데이터 업로드 & 형식 변환
왼쪽 파일 패널에 `cora.jsonl`(또는 선택한 에이전트 파일)을 올리거나, 아래 셀로 업로드.

In [ ]:
import os
if not os.path.exists(DATA_FILE):
    from google.colab import files
    print(f"⬆️ {DATA_FILE} 파일을 선택해 업로드하세요")
    files.upload()

from datasets import load_dataset
dataset = load_dataset("json", data_files=DATA_FILE, split="train")
print(f"✅ {len(dataset)}개 로드")
print(dataset[0])

In [ ]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)  # 'conversations' 표준화

def formatting(examples):
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in examples["conversations"]
    ]
    return {"text": texts}

dataset = dataset.map(formatting, batched=True)
print(dataset[0]["text"][:400])

## 5. LoRA 어댑터 부착
전체 파라미터 중 약 0.3%만 학습 — 저비용으로 두뇌 특화.

In [ ]:
model = FastModel.get_peft_model(
    model,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    random_state = 3407,
)

## 6. 학습
**train_on_responses_only** = 질문은 빼고 **답변(코라의 말)만** 학습 → 함정②(echo) 예방.

| 파라미터 | 의미 | 시작값 |
| --- | --- | --- |
| learning_rate | 학습 속도 | 2e-4 |
| max_steps | 학습 횟수 | 60 (데이터 30개 기준) |
| batch×accum | 유효 배치 | 2×4=8 |

🎯 목표: **Loss 0.2~0.5**. `>1.0`이면 step↑, `<0.01`이면 과적합이니 step↓.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,            # ← 데이터/Loss 보며 조절
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

In [ ]:
trainer_stats = trainer.train()
# 마지막 로그의 'loss' 값이 0.2~0.5 근처인지 확인!

## 7. ✅ 테스트 — 진짜 그 페르소나가 됐는지
학습 전 답변과 비교해보자. 정체성 + 전문 질문 둘 다 확인.

In [ ]:
tests = {
    "cora":  ["넌 누구야?", "첫 문장이 안 써져"],
    "finn":  ["넌 누구야?", "AI로 돈 벌기 뭐부터 시작해?"],
    "offie": ["넌 누구야?", "일이 너무 많아서 숨이 막혀"],
    "rina":  ["넌 누구야?", "사람이 잘 안 모여서 속상해"],
    "geo":   ["넌 누구야?", "하고 싶은 게 너무 많아서 뭐부터 해야 할지 모르겠어"],
}
for q in tests.get(AGENT, ["넌 누구야?"]):
    print(f"Q: {q}\nA: {ask(model, q)}\n")

**결과 해석**
- 페르소나 말투/시그니처가 나오면 → 🎉 성공
- 질문을 그대로 따라 하거나(echo) 밋밋하면 → 함정② 과적합/데이터 부족 → `max_steps↓`, 데이터 50개+로 늘리기
- 말투는 없는데 Loss는 낮았다면 → 함정① `<bos>` 형식 불일치 점검

## 8. 저장
LoRA 어댑터만 저장(가볍다). 다음 주: LM Studio/Ollama 연결용 GGUF 변환.

In [ ]:
save_dir = f"{AGENT}_lora"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"💾 저장: {save_dir}")

# 압축 다운로드
import shutil
shutil.make_archive(save_dir, "zip", save_dir)
from google.colab import files
files.download(f"{save_dir}.zip")

# (선택) Hugging Face 업로드 — 토큰 필요
# model.push_to_hub_merged("내아이디/cora", tokenizer, token="hf_...")
#
# ⚠️ 함정③: GGUF 변환은 GPU가 아니라 '시스템 RAM' 부족으로 커널이 죽는다(T4 12.7GB < ~15GB).
#   → Colab Pro(고RAM) 사용하거나, HF에 merged로 올린 뒤 변환하기.